In [32]:
# ============================================================
# 로컬/PyCharm ipynb용 프레임 확인 도구
# HTML video 사용 안 함
# OpenCV + ipywidgets 방식
# ============================================================

from pathlib import Path
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------------------
# 1. 여기만 수정
# ------------------------------------------------------------
BASE_DIR = Path("data/측면/data")
VIDEO_REL_PATH = "deadlift/stu8_4.mp4"

VIDEO_PATH = BASE_DIR / VIDEO_REL_PATH

if not VIDEO_PATH.exists():
    print("영상을 찾지 못했습니다.")
    print("입력한 경로:", VIDEO_PATH)

    video_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
    found = []

    if BASE_DIR.exists():
        for p in BASE_DIR.rglob("*"):
            if p.is_file() and p.suffix.lower() in video_exts:
                found.append(p.relative_to(BASE_DIR))

    print("\n현재 base 폴더 아래 영상 일부:")
    for p in found[:30]:
        print(p)

    raise FileNotFoundError(VIDEO_PATH)

# ------------------------------------------------------------
# 2. 영상 정보
# ------------------------------------------------------------
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(f"영상 열기 실패: {VIDEO_PATH}")

fps = float(cap.get(cv2.CAP_PROP_FPS))
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = num_frames / fps if fps > 0 else 0

print("선택한 영상:", VIDEO_PATH)
print("fps:", fps)
print("num_frames:", num_frames)
print("duration(sec):", duration)
print("resolution:", width, "x", height)

# ------------------------------------------------------------
# 3. 프레임 읽기 함수
# ------------------------------------------------------------
def read_frame(frame_idx):
    frame_idx = max(0, min(frame_idx, num_frames - 1))

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        raise RuntimeError(f"프레임 읽기 실패: {frame_idx}")

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 표시용 축소
    max_width = 760
    if frame.shape[1] > max_width:
        scale = max_width / frame.shape[1]
        new_w = int(frame.shape[1] * scale)
        new_h = int(frame.shape[0] * scale)
        frame = cv2.resize(frame, (new_w, new_h))

    ok, encoded = cv2.imencode(".jpg", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    if not ok:
        raise RuntimeError("프레임 jpg 인코딩 실패")

    return encoded.tobytes()

# ------------------------------------------------------------
# 4. 위젯 구성
# ------------------------------------------------------------
current_frame = 0
start_frame = None
end_frame = None

image_widget = widgets.Image(
    value=read_frame(current_frame),
    format="jpg",
    width=760
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_frames - 1,
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="760px")
)

info_label = widgets.HTML()
rep_label = widgets.HTML()
log_output = widgets.Output()

btn_prev = widgets.Button(description="◀ -1 frame")
btn_next = widgets.Button(description="+1 frame ▶")
btn_print = widgets.Button(description="F: Print Frame")
btn_start = widgets.Button(description="A: Mark Start")
btn_end = widgets.Button(description="D: Mark End")
btn_reset = widgets.Button(description="Reset")

# ------------------------------------------------------------
# 5. 업데이트 함수
# ------------------------------------------------------------
def update_display(frame_idx):
    global current_frame

    current_frame = max(0, min(frame_idx, num_frames - 1))
    image_widget.value = read_frame(current_frame)

    cur_time = current_frame / fps if fps > 0 else 0

    info_label.value = f"""
    <b>video:</b> {VIDEO_REL_PATH}<br>
    <b>fps:</b> {fps:.6f}<br>
    <b>num_frames:</b> {num_frames}<br>
    <b>current frame:</b> {current_frame}<br>
    <b>current time:</b> {cur_time:.3f} sec<br>
    """

    l1 = "" if start_frame is None else start_frame
    l2 = "" if end_frame is None else end_frame

    rep_label.value = f"""
    <b>Start frame:</b> {start_frame}<br>
    <b>End frame:</b> {end_frame}<br>
    <b>RepCount format:</b>
    <pre>L1={l1}, L2={l2}</pre>
    """

def log(msg):
    with log_output:
        print(msg)

# ------------------------------------------------------------
# 6. 이벤트 함수
# ------------------------------------------------------------
def on_slider_change(change):
    if change["name"] == "value":
        update_display(change["new"])

def step(delta):
    new_frame = max(0, min(current_frame + delta, num_frames - 1))
    frame_slider.value = new_frame
    update_display(new_frame)

def on_prev_clicked(b):
    step(-1)

def on_next_clicked(b):
    step(1)

def on_print_clicked(b):
    cur_time = current_frame / fps if fps > 0 else 0
    log(f"현재 위치: time={cur_time:.3f} sec, frame={current_frame}")

def on_start_clicked(b):
    global start_frame
    start_frame = current_frame
    log(f"START 설정: L1 = {start_frame}")
    update_display(current_frame)

def on_end_clicked(b):
    global end_frame
    end_frame = current_frame
    log(f"END 설정: L2 = {end_frame}")

    if start_frame is not None and end_frame is not None:
        s, e = sorted([start_frame, end_frame])
        log(f"라벨 결과: count=1, L1={s}, L2={e}")

    update_display(current_frame)

def on_reset_clicked(b):
    global start_frame, end_frame
    start_frame = None
    end_frame = None

    with log_output:
        clear_output()

    update_display(current_frame)

frame_slider.observe(on_slider_change)
btn_prev.on_click(on_prev_clicked)
btn_next.on_click(on_next_clicked)
btn_print.on_click(on_print_clicked)
btn_start.on_click(on_start_clicked)
btn_end.on_click(on_end_clicked)
btn_reset.on_click(on_reset_clicked)

# ------------------------------------------------------------
# 7. 표시
# ------------------------------------------------------------
update_display(current_frame)

display(info_label)
display(image_widget)
display(frame_slider)

display(widgets.HBox([
    btn_prev,
    btn_next,
    btn_print,
    btn_start,
    btn_end,
    btn_reset
]))

display(rep_label)
display(log_output)

선택한 영상: data\측면\data\deadlift\stu8_4.mp4
fps: 29.97002997002997
num_frames: 960
duration(sec): 32.032000000000004
resolution: 640 x 360


HTML(value='\n    <b>video:</b> deadlift/stu8_4.mp4<br>\n    <b>fps:</b> 29.970030<br>\n    <b>num_frames:</b>…

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

IntSlider(value=0, continuous_update=False, description='Frame:', layout=Layout(width='760px'), max=959)

HTML(value='\n    <b>Start frame:</b> None<br>\n    <b>End frame:</b> None<br>\n    <b>RepCount format:</b>\n …

Output()